In [1]:
# Colab compatibility: install packages not preinstalled there (no-op locally / on Colab reruns)
import importlib.util, subprocess, sys
for pkg, mod in [('faiss-cpu', 'faiss')]:
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# 04b - Hard-negative mining (bonus extension)

In notebook 04 the two-tower trains on in-batch negatives: for each user, the other items in the batch count as negatives. Those are basically random items, and random items are easy. The model learns to tell them apart fast and then stops getting much signal from them.

Hard negatives are the opposite. They're items the trained model ranks highly (FAISS ranks around 50-200) that the user never actually interacted with, so they're its confident mistakes. Feeding those back in as negatives makes it work on the distinctions it's currently getting wrong.

The steps: train v2 in notebook 04, use it to mine each user's FAISS ranks 50-200, retrain from scratch with those added to the softmax denominator, then compare v2 against v2+HN on test. Nothing leaks here, since the negatives come only from train-period retrieval.

## 0. Setup (same data and config as notebook 04)

In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
import json, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import faiss

SEED = 42; torch.manual_seed(SEED); np.random.seed(SEED); rng = np.random.default_rng(SEED)
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('Device:', device)

try:
    from google.colab import drive; drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/recsys_kindle/data'
except ImportError:
    DATA_DIR = './data'
ART = os.path.join(DATA_DIR, 'artifacts_two_tower')

train = pd.read_parquet(f'{DATA_DIR}/train.parquet')
val   = pd.read_parquet(f'{DATA_DIR}/val.parquet')
test  = pd.read_parquet(f'{DATA_DIR}/test.parquet')
mappings = json.load(open(f'{DATA_DIR}/id_mappings.json'))
N_ITEMS = len(mappings['item2idx'])

DIM, HIDDEN, MAX_HIST, PAD = 128, 256, 20, N_ITEMS   # match notebook 04's v2
TAU, USE_LOGQ, GAMMA = 0.1, True, 0.85
BATCH = 8192 if torch.cuda.is_available() else 4096
H_NEG = 8      # hard negatives sampled per training example
EPOCHS = 8
LO, HI = 50, 200   # FAISS rank band to mine (see markdown)

pos_sorted = train[train.is_positive].sort_values(['user_idx', 'timestamp'])
user_seqs = {int(u): g.item_idx.to_numpy(np.int64) for u, g in pos_sorted.groupby('user_idx')}
seen_train = train.groupby('user_idx')['item_idx'].agg(set).to_dict()
ex_users, ex_pos = [], []
for u, seq in user_seqs.items():
    if len(seq) >= 2:
        ex_users.append(np.full(len(seq) - 1, u)); ex_pos.append(np.arange(1, len(seq)))
ex_users = np.concatenate(ex_users); ex_pos = np.concatenate(ex_pos)
tc = np.bincount(pos_sorted.item_idx.to_numpy(np.int64), minlength=N_ITEMS)
logQ = torch.tensor(np.log(np.maximum(tc, 1) / tc.sum()), dtype=torch.float32, device=device)

def build_truth(p):
    p = p[p.is_positive & p.user_idx.notna() & p.item_idx.notna()]
    return p.groupby('user_idx')['item_idx'].agg(set).to_dict()
truth_val, truth_test = build_truth(val), build_truth(test)
print(f'{len(ex_users):,} training examples, {N_ITEMS:,} items')

Device: mps


1,912,936 training examples, 120,573 items


## 1. Model + evaluation (identical to notebook 04)

In [3]:
class TwoTower(nn.Module):
    def __init__(self, n, dim, hid, gamma=1.0):
        super().__init__()
        self.emb = nn.Embedding(n + 1, dim, padding_idx=n)
        nn.init.normal_(self.emb.weight, std=0.01)
        with torch.no_grad(): self.emb.weight[n].zero_()
        self.user_mlp = nn.Sequential(nn.Linear(dim, hid), nn.ReLU(), nn.Linear(hid, dim))
        self.item_mlp = nn.Sequential(nn.Linear(dim, hid), nn.ReLU(), nn.Linear(hid, dim))
        self.gamma = gamma
    def user_tower(self, hist, mask):
        pos = mask.cumsum(1); n = mask.sum(1, keepdim=True); dist = (n - pos).clamp(min=0)
        w = (self.gamma ** dist) * mask; w = w / w.sum(1, keepdim=True).clamp(min=1e-8)
        return F.normalize(self.user_mlp((self.emb(hist) * w.unsqueeze(-1)).sum(1)), dim=-1)
    def item_tower(self, items):
        return F.normalize(self.item_mlp(self.emb(items)), dim=-1)

@torch.no_grad()
def all_item_emb(m, batch=65536):
    m.eval(); outs = []
    for s in range(0, N_ITEMS, batch):
        ids = torch.arange(s, min(s + batch, N_ITEMS), device=device)
        outs.append(m.item_tower(ids).cpu().numpy())
    return np.vstack(outs).astype('float32')
@torch.no_grad()
def embed_users(m, users):
    hist = np.full((len(users), MAX_HIST), PAD, np.int64)
    for r, u in enumerate(users):
        seq = user_seqs.get(u, np.array([], np.int64))[-MAX_HIST:]; hist[r, :len(seq)] = seq
    h = torch.as_tensor(hist, device=device)
    return m.user_tower(h, (h != PAD).float()).cpu().numpy().astype('float32')
def retrieve(m, index, users, k=100, batch=8192):
    recs = {}; users = [u for u in users if u in user_seqs]
    for s in range(0, len(users), batch):
        ch = users[s:s+batch]; buf = max((len(seen_train.get(u, ())) for u in ch), default=0) + 10
        _, I = index.search(embed_users(m, ch), k + buf)
        for u, row in zip(ch, I):
            recs[u] = [int(i) for i in row if i not in seen_train.get(u, set())][:k]
    return recs
def recall_at_k(recs, truth, k):
    s = [len(set(recs.get(u, [])[:k]) & rel) / len(rel) for u, rel in truth.items() if rel]
    return float(np.mean(s)) if s else 0.0
def ndcg_at_k(recs, truth, k):
    out = []
    for u, rel in truth.items():
        top = recs.get(u, [])[:k]
        dcg = sum(1 / np.log2(i + 2) for i, it in enumerate(top) if it in rel)
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(rel), k)))
        out.append(dcg / idcg if idcg > 0 else 0.0)
    return float(np.mean(out))
def coverage(recs, k):
    s = set(); [s.update(r[:k]) for r in recs.values()]; return len(s) / N_ITEMS
def eval_test(m):
    idx = faiss.IndexFlatIP(DIM); idx.add(all_item_emb(m))
    r = retrieve(m, idx, truth_test.keys(), 100)
    return {'Recall@20': recall_at_k(r, truth_test, 20), 'Recall@50': recall_at_k(r, truth_test, 50),
            'NDCG@10': ndcg_at_k(r, truth_test, 10), 'Coverage@10': coverage(r, 10)}

## 2. Mine hard negatives with the trained v2 model

Load v2 (exported by notebook 04), retrieve each user's top-200, and keep ranks 50 to 200 after dropping anything they've already seen in training.

Why that band? Below rank 50 you risk grabbing items the user would actually like but hasn't reached yet. Those are false negatives, and training against them just makes the model worse. Above 200 the items are easy again, already scored low, so there's nothing to learn from them. Ranks 50 to 200 are the sweet spot: the model is confident about them, but they're almost certainly genuine non-interests. Those are the mistakes worth training on.

In [4]:
faiss.omp_set_num_threads(1)
v2 = TwoTower(N_ITEMS, DIM, HIDDEN, GAMMA).to(device)
v2.load_state_dict(torch.load(f'{ART}/two_tower.pt', map_location=device)); v2.eval()
v2_res = eval_test(v2)
print('v2 baseline (test):', {k: round(x, 4) for k, x in v2_res.items()})

index = faiss.IndexFlatIP(DIM); index.add(all_item_emb(v2))
train_users = [u for u in user_seqs if len(user_seqs[u]) >= 2]
hard = {}; t0 = time.time()
for s in range(0, len(train_users), 8192):
    ch = train_users[s:s+8192]; _, I = index.search(embed_users(v2, ch), HI)
    for u, row in zip(ch, I):
        seen = seen_train.get(u, set())
        negs = [int(i) for i in row[LO:HI] if i not in seen]
        if negs: hard[u] = np.array(negs, np.int64)
print(f'mined hard negatives for {len(hard):,} users in {time.time()-t0:.0f}s')

v2 baseline (test): {'Recall@20': 0.0572, 'Recall@50': 0.0967, 'NDCG@10': 0.0215, 'Coverage@10': 0.2996}


mined hard negatives for 121,995 users in 16s


## 3. Hard-negative loss

Each user's positive item now has to beat the B-1 in-batch negatives plus H_NEG of that user's own mined hard negatives. In code that's just an extra [B, H] block of scores glued onto the in-batch logits. The log-Q correction and false-negative masking from notebook 04 carry over unchanged.

In [5]:
def make_batch_hn(idx):
    B = len(idx)
    hist = np.full((B, MAX_HIST), PAD, np.int64); target = np.empty(B, np.int64); hn = np.zeros((B, H_NEG), np.int64)
    for r, e in enumerate(idx):
        u, k = ex_users[e], ex_pos[e]; seq = user_seqs[u]; h = seq[max(0, k - MAX_HIST):k]
        hist[r, :len(h)] = h; target[r] = seq[k]
        pool = hard.get(u)
        if pool is not None and len(pool):
            hn[r] = rng.choice(pool, size=H_NEG, replace=len(pool) < H_NEG)
        else:
            hn[r] = rng.integers(0, N_ITEMS, size=H_NEG)   # fallback for users with no mined pool
    ht = torch.as_tensor(hist, device=device)
    return ht, (ht != PAD).float(), torch.as_tensor(target, device=device), torch.as_tensor(hn, device=device)

def loss_hn(m, hist, mask, target, hn):
    B = len(target)
    u = m.user_tower(hist, mask); v = m.item_tower(target)
    in_logits = (u @ v.T) / TAU                                    # [B, B] in-batch negatives
    v_hn = m.item_tower(hn.reshape(-1)).reshape(B, H_NEG, DIM)
    hn_logits = torch.einsum('bd,bhd->bh', u, v_hn) / TAU          # [B, H] per-user hard negatives
    if USE_LOGQ:
        in_logits = in_logits - logQ[target].unsqueeze(0)
        hn_logits = hn_logits - logQ[hn]
    same = target.unsqueeze(0) == target.unsqueeze(1); eye = torch.eye(B, dtype=torch.bool, device=device)
    in_logits = in_logits.masked_fill(same & ~eye, float('-inf'))
    logits = torch.cat([in_logits, hn_logits], dim=1)             # [B, B + H]
    return F.cross_entropy(logits, torch.arange(B, device=device))

## 4. Retrain from scratch with hard negatives

In [6]:
hn_model = TwoTower(N_ITEMS, DIM, HIDDEN, GAMMA).to(device)
opt = torch.optim.Adam(hn_model.parameters(), lr=1e-3)
n = len(ex_users)
for ep in range(1, EPOCHS + 1):
    hn_model.train(); t0 = time.time(); tot = 0.0; perm = rng.permutation(n)
    for s in range(0, n, BATCH):
        hist, mask, target, hn = make_batch_hn(perm[s:s+BATCH])
        l = loss_hn(hn_model, hist, mask, target, hn)
        opt.zero_grad(); l.backward(); opt.step(); tot += l.item() * len(target)
    print(f'HN epoch {ep} | loss {tot/n:.4f} | {time.time()-t0:.0f}s')

HN epoch 1 | loss 7.1862 | 27s


HN epoch 2 | loss 5.3370 | 27s


HN epoch 3 | loss 4.7703 | 27s


HN epoch 4 | loss 4.4885 | 27s


HN epoch 5 | loss 4.3081 | 27s


HN epoch 6 | loss 4.1789 | 30s


HN epoch 7 | loss 4.0798 | 34s


HN epoch 8 | loss 3.9989 | 37s


## 5. Before / after (test)

In [7]:
hn_res = eval_test(hn_model)
cmp = pd.DataFrame({'v2 (in-batch only)': v2_res, 'v2 + hard negatives': hn_res})
cmp['delta'] = cmp['v2 + hard negatives'] - cmp['v2 (in-batch only)']
cmp['delta %'] = (100 * cmp['delta'] / cmp['v2 (in-batch only)']).round(1)
display(cmp.round(4))
cmp.round(4).to_json(os.path.join(DATA_DIR, 'hard_negatives_results.json'), indent=2)
print('winner on Recall@20:', 'v2 + HN' if hn_res['Recall@20'] > v2_res['Recall@20'] else 'v2 (kept as deployed model)')

,v2 (in-batch only),v2 + hard negatives,delta,delta %
Recall@20,0.0572,0.0537,-0.0035,-6.1
Recall@50,0.0967,0.0929,-0.0038,-4.0
NDCG@10,0.0215,0.0201,-0.0014,-6.3
Coverage@10,0.2996,0.3376,0.0381,12.7


winner on Recall@20: v2 (kept as deployed model)


**Interpretation.** Hard negatives didn't win: Recall@20 and NDCG@10 both drop about 6%, coverage rises +13%. On a sparse catalog, FAISS ranks 50-200 are mostly unobserved positives (books the user would like but never rated), so training on them as negatives pushes away relevant items. I'm keeping v2. A negative result I can explain beats a win I can't.